In [1]:
# %pip install torch transformers huggingface_hub pillow auto-round optimum gptqmodel compressed-tensors --break-system-packages

In [2]:
import os
import sys

import torch
from PIL import Image, ImageDraw


In [3]:
def create_dummy_receipt_image(filename="sample_receipt.png"):
    """Creates a temporary synthetic image to test multimodal evaluation."""
    img = Image.new("RGB", (360, 180), color=(250, 250, 250))
    draw = ImageDraw.Draw(img)
    draw.text((20, 20), "ACME SUPPLIES STORE", fill=(0, 0, 0))
    draw.text((20, 50), "Item: Heavy Duty Widget x2", fill=(0, 0, 0))
    draw.text((20, 80), "Subtotal: $1,200.00", fill=(0, 0, 0))
    draw.text((20, 110), "Status: UNPAID / OVERDUE", fill=(200, 0, 0))
    draw.text((20, 140), "Total: $1,200.00", fill=(0, 0, 0))
    img.save(filename)
    return filename


In [4]:
MODEL_ID = "Vishva007/clef-flash-W4A16-AutoRound-GPTQ"
LOCAL_PATH = "./local_model-AutoRound-GPTQ"

!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 15 files: 100%|██████████████████████| 15/15 [00:00<00:00, 904.99it/s]
Download complete: :                                       |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model-AutoRound-GPTQ
Download complete: :                                       |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [5]:
# 1. Add your local model directory to Python's module search path
model_dir = os.path.abspath(LOCAL_PATH)
if model_dir not in sys.path:
    sys.path.insert(0, model_dir)

# 2. Now Python will find local_model/joint_schema_model.py
from joint_schema_model import load_release_model, systemone

# 3. Load the model from your local path
model, processor = load_release_model(LOCAL_PATH, device="cuda")

W1004 04:33:47.337000 15072 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1004 04:33:47.354000 15072 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.



INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.5.0
Transformers : 5.17.0
Torch        : 2.14.0+cu132
Triton       : 3.8.0
INFO  Kernel: Auto-selection: adding candidate `MarlinLinear`                   
INFO  K

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Loading weights:   0%|          | 0/1504 [00:00<?, ?it/s]

WARN  Marlin is padding a tile-misaligned weight shape. Activations and outputs for this layer are padded and sliced on each forward; performance may be degraded.
INFO  gc.collect() reclaimed 26 objects in 0.260s                               


In [6]:
def run_test_cases():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading Clef-Flash on {device.upper()} (BF16)...")
    # -------------------------------------------------------------
    # Test Case 1: Plain Text (Customer Support & Incident Triage)
    # -------------------------------------------------------------
    test_1 = {
        "model": "clef-flash",
        "state": (
            "Prod database latency surged past 4,000ms after the 2:00 PM release. "
            "Users cannot checkout, cart operations are timing out with HTTP 504 errors."
        ),
        "questions": {
            "incident_level": {
                "type": "choice",
                "instructions": "Determine the severity tier of this incident.",
                "criteria": {
                    "SEV_1": "Critical outage affecting revenue or all users.",
                    "SEV_2": "Major feature degradation with no immediate workaround.",
                    "SEV_3": "Minor bug or cosmetic glitch with minimal impact.",
                },
            },
            "urgency": {
                "type": "score",
                "instructions": "Estimate resolution timeframe needed.",
                "criteria": [
                    "Can wait for sprint",
                    "Within 24 hours",
                    "Immediate page",
                ],
            },
            "requires_rollback": {
                "type": "noul",
                "instructions": "Does this require rolling back the release?",
            },
        },
    }

    # -------------------------------------------------------------
    # Test Case 2: Structured JSON Input (Invoice Risk Evaluation)
    # -------------------------------------------------------------
    test_2 = {
        "model": "clef-flash",
        "state": {
            "invoice_id": "INV-2026-9041",
            "vendor": "Acme Cloud Services",
            "amount_usd": 15400.00,
            "due_date": "2026-08-15",
            "current_date": "2026-10-03",
            "historical_defaults": 0,
            "flagged_suspicious": False,
        },
        "questions": {
            "approval_action": {
                "type": "choice",
                "instructions": "What action should the AP engine execute?",
                "criteria": {
                    "auto_approve": "Standard payment processing under threshold.",
                    "escalate_director": "Requires director sign-off due to size or overdue state.",
                    "reject_fraud": "Invoice appears fraudulent or forged.",
                },
            },
            "is_past_due": {
                "type": "noul",
                "instructions": "Is the invoice significantly overdue?",
            },
        },
    }

    # -------------------------------------------------------------
    # Test Case 3: Multimodal (Receipt Image + Text Prompt)
    # -------------------------------------------------------------
    sample_img_path = create_dummy_receipt_image()
    test_3 = {
        "model": "clef-flash",
        "state": "Please inspect this uploaded accounting receipt.",
        "images": [Image.open(sample_img_path)],
        "questions": {
            "document_clarity": {
                "type": "choice",
                "instructions": "Assess visual document quality.",
                "criteria": {
                    "legible": "Clear text and amounts visible.",
                    "blurry": "Partially readable but degraded.",
                    "unreadable": "Too dark, corrupted, or occluded.",
                },
            },
            "exceeds_1000_usd": {
                "type": "noul",
                "instructions": "Is the total value displayed greater than $1,000 USD?",
            },
        },
    }

    test_suite = [
        ("Test 1: Plain Text Triage", test_1),
        ("Test 2: Structured JSON State", test_2),
        ("Test 3: Multimodal Image", test_3),
    ]

    for title, payload in test_suite:
        print("\n" + "=" * 60)
        print(f"RUNNING: {title}")
        print("=" * 60)

        # systemone handles inference & returns formatted answers
        result = systemone(model, processor, payload)
        answers = result["answers"]

        for q_id, q_data in answers.items():
            print(f"\n[Question ID: '{q_id}']")

            # Choice questions
            if "choice" in q_data:
                print(f"  Selected Choice: {q_data['choice']} (Confidence: {q_data.get('confidence', 0):.3f})")
                print("  Probabilities:")
                for opt, p in q_data.get("probabilities", {}).items():
                    print(f"    - {opt}: {float(p):.4f}")

            # Score questions (probabilities is a dict mapping level indices to probabilities)
            elif "score" in q_data:
                print(f"  Expected Score : {q_data['score']:.2f}")
                print(f"  Legend / Scale : {q_data.get('legend')}")
                print(f"  Confidence     : {q_data.get('confidence', 0):.3f}")
                probs = q_data.get("probabilities", {})
                if isinstance(probs, dict):
                    print("  Probabilities  :")
                    for lvl, p in probs.items():
                        desc = q_data.get("legend", {}).get(str(lvl), "")
                        print(f"    - Level {lvl} ({desc}): {float(p):.4f}")
                elif isinstance(probs, list):
                    print(f"  Probabilities  : {[round(float(x), 4) for x in probs]}")

            # Noul (binary) questions
            elif "noul" in q_data:
                p_true = float(q_data["noul"])
                print(f"  Binary (True/False): {p_true > 0.5} (p_true = {p_true:.4f})")
            elif isinstance(q_data, (float, int)):
                p_true = float(q_data)
                print(f"  Binary (True/False): {p_true > 0.5} (p_true = {p_true:.4f})")
            else:
                print(f"  Raw: {q_data}")


In [7]:
if __name__ == "__main__":
    run_test_cases()

Loading Clef-Flash on CUDA (BF16)...

RUNNING: Test 1: Plain Text Triage


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.



[Question ID: 'incident_level']
  Selected Choice: SEV_1 (Confidence: 0.912)
  Probabilities:
    - SEV_1: 0.9118
    - SEV_2: 0.0743
    - SEV_3: 0.0140

[Question ID: 'urgency']
  Expected Score : 1.93
  Legend / Scale : {'0': 'Can wait for sprint', '1': 'Within 24 hours', '2': 'Immediate page'}
  Confidence     : 0.944
  Probabilities  :
    - Level 0 (Can wait for sprint): 0.0151
    - Level 1 (Within 24 hours): 0.0411
    - Level 2 (Immediate page): 0.9437

[Question ID: 'requires_rollback']
  Binary (True/False): True (p_true = 0.8222)

RUNNING: Test 2: Structured JSON State

[Question ID: 'approval_action']
  Selected Choice: escalate_director (Confidence: 0.961)
  Probabilities:
    - auto_approve: 0.0249
    - escalate_director: 0.9605
    - reject_fraud: 0.0146

[Question ID: 'is_past_due']
  Binary (True/False): True (p_true = 0.7280)

RUNNING: Test 3: Multimodal Image

[Question ID: 'document_clarity']
  Selected Choice: legible (Confidence: 0.962)
  Probabilities:
    - l